# 🎓 T-GNN: Temporal Heterogeneous GNN for Academic Credential Fraud Detection

**Full experiment pipeline for paper review:** baselines · ablation · sensitivity · figures

> ⚠️ **Before running:** Set **Runtime → Change runtime type → T4 GPU** then click **Run all**.

## Phase 1 — Install Dependencies

In [ ]:
!nvidia-smi 2>/dev/null || echo 'No GPU — training will use CPU (slower)'
import torch
print(f'PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}')

TORCH_VER = torch.__version__.split('+')[0]
CUDA_TAG  = 'cu121' if torch.cuda.is_available() else 'cpu'

!pip install -q torch-geometric
!pip install -q pyg-lib torch-scatter torch-sparse \
    -f https://data.pyg.org/whl/torch-{TORCH_VER}+{CUDA_TAG}.html 2>/dev/null || true
!pip install -q pyarrow pyyaml tqdm seaborn

print('\n✅ All dependencies installed')

## Phase 2 — Upload Codebase

Click **Choose Files** and upload your `TGNN-code.zip`.

In [ ]:
import zipfile, os
from google.colab import files

print('Upload your TGNN-code.zip file:')
uploaded = files.upload()

zip_name = list(uploaded.keys())[0]
extract_path = '/content/'

with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall(extract_path)

# Auto-detect extracted folder name
REPO = None

# 1. Try to use the zip file's base name as the directory name
guessed_repo_dir_name = os.path.splitext(zip_name)[0] # e.g. 'TGNN-code'
potential_repo_path = os.path.join(extract_path, guessed_repo_dir_name)
if os.path.isdir(potential_repo_path):
    REPO = potential_repo_path
elif os.path.isdir(os.path.join(extract_path, 'TGNN-code')):
    # 2. Fallback to a common default 'TGNN-code' if it exists
    REPO = os.path.join(extract_path, 'TGNN-code')
else:
    # 3. Search for any directory in /content that contains 'TGNN' or 'tgnn'
    found_repo_dir = None
    for item in os.listdir(extract_path):
        item_path = os.path.join(extract_path, item)
        if os.path.isdir(item_path) and ('TGNN' in item or 'tgnn' in item.lower()):
            found_repo_dir = item_path
            break
    if found_repo_dir:
        REPO = found_repo_dir
    else:
        # 4. If none of the above identify a specific subdirectory,
        # it implies the contents were extracted directly into /content/.
        # In this case, the repository root is '/content/' itself.
        REPO = extract_path

# Ensure REPO is not None (should be covered by the last else)
if REPO is None:
    REPO = extract_path

print(f'Codebase at: {REPO}')
os.listdir(REPO)

## Phase 3 — Validate Synthetic Dataset

In [ ]:
import os

DATA_DIR = os.path.join(REPO, 'data/synthetic/')
parquet_files = ['events.parquet', 'students.parquet', 'institutions.parquet',
                 'verifiers.parquet', 'credentials.parquet']
already_exists = all(os.path.exists(os.path.join(DATA_DIR, f)) for f in parquet_files)

if already_exists:
    print('✅ Synthetic data already in ZIP — skipping generation')
else:
    print('Generating synthetic dataset (seed=42) …')
    !python data/generate_synthetic.py --seed 42 --config configs/default.yaml
    print('✅ Generation complete')

In [ ]:
!python data/validate_dataset.py --config configs/default.yaml

## Phase 4 — Baseline Comparisons

Models: **Static GCN · CNN-1D · Isolation Forest · TGAT · TGN · T-GNN (ours)**  
Seeds: 42, 123, 2024, 3407, 7777

In [ ]:
import subprocess, sys, time

os.makedirs('results/logs', exist_ok=True)
t0 = time.time()
print('Starting baseline experiments …')

ret = subprocess.run(
    [sys.executable, 'experiments/run_baselines.py',
     '--seeds', '42', '123', '2024', '3407', '7777',
     '--config', 'configs/default.yaml'],
    cwd=REPO
)
elapsed = time.time() - t0
status = '✅ Done' if ret.returncode == 0 else f'❌ Failed (code {ret.returncode})'
print(f'\n{status} — {elapsed/60:.1f} min')

In [ ]:
import json, glob, pandas as pd

records = []
for fp in glob.glob(os.path.join(REPO, 'results/raw/baselines/*.json')):
    with open(fp) as f:
        records.append(json.load(f))

df_bl = pd.DataFrame(records)
MODEL_ORDER = ['static_gcn', 'cnn', 'isolation_forest', 'tgat', 'tgn', 'tgnn']
summary = df_bl.groupby('model').agg(
    Precision_mean=('test_precision', 'mean'),
    Recall_mean   =('test_recall',    'mean'),
    F1_mean       =('test_f1',        'mean'),
    F1_std        =('test_f1',        'std'),
    AUC_mean      =('test_roc_auc',   'mean'),
    AUC_std       =('test_roc_auc',   'std'),
).reindex([m for m in MODEL_ORDER if m in df_bl['model'].unique()]).round(4)

print('=== BASELINE SUMMARY (mean ± std, 5 seeds) ===')
print(summary.to_string())
summary

## Phase 5 — Ablation Study

| Config | Description |
|--------|-------------|
| A1 | Static GCN — no GRU, no attention |
| A2 | GCN + GRU — no relation attention |
| A3 | GCN + Relation Attention — no GRU |
| A4 | Temporal + Attention — homogeneous |
| A5 | **Full T-GNN (proposed)** |

In [ ]:
t0 = time.time()
print('Starting ablation study …')
ret = subprocess.run(
    [sys.executable, 'experiments/run_ablation.py',
     '--seeds', '42', '123', '2024', '3407', '7777',
     '--config', 'configs/default.yaml'],
    cwd=REPO
)
elapsed = time.time() - t0
status = '✅ Done' if ret.returncode == 0 else f'❌ Failed (code {ret.returncode})'
print(f'\n{status} — {elapsed/60:.1f} min')

In [ ]:
records_abl = []
for fp in glob.glob(os.path.join(REPO, 'results/raw/ablation/*.json')):
    with open(fp) as f:
        records_abl.append(json.load(f))

df_abl = pd.DataFrame(records_abl)
ABL_ORDER = ['A1_static_gcn', 'A2_gcn_gru', 'A3_gcn_attention',
             'A4_temporal_attention_no_hetero', 'A5_full_tgnn']
summary_abl = df_abl.groupby('config').agg(
    F1_mean =("test_f1",      'mean'),
    F1_std  =("test_f1",      'std'),
    AUC_mean=("test_roc_auc", 'mean'),
    AUC_std =("test_roc_auc", 'std'),
).reindex([c for c in ABL_ORDER if c in df_abl['config'].unique()]).round(4)

print('=== ABLATION SUMMARY (mean ± std, 5 seeds) ===')
print(summary_abl.to_string())
summary_abl

## Phase 6 — Sensitivity Analysis

Sweeps: **embedding dim** [16,32,64,128,192,256,512] · **snapshot window** [1,5,10,15,20,30,45,60] days

In [ ]:
t0 = time.time()
print('Starting sensitivity analysis …')
ret = subprocess.run(
    [sys.executable, 'experiments/run_sensitivity.py',
     '--seeds', '42', '123',
     '--config', 'configs/default.yaml'],
    cwd=REPO
)
elapsed = time.time() - t0
status = '✅ Done' if ret.returncode == 0 else f'❌ Failed (code {ret.returncode})'
print(f'\n{status} — {elapsed/60:.1f} min')

In [ ]:
records_sens = []
for fp in glob.glob(os.path.join(REPO, 'results/raw/sensitivity/*.json')):
    with open(fp) as f:
        records_sens.append(json.load(f))

df_sens = pd.DataFrame(records_sens)
print('=== SENSITIVITY — Embedding Dim (F1) ===')
print(df_sens[df_sens.sweep == 'embedding_dim']
      .groupby('value')['f1'].agg(['mean', 'std']).round(4).to_string())
print('\n=== SENSITIVITY — Snapshot Window (F1) ===')
print(df_sens[df_sens.sweep == 'snapshot_window']
      .groupby('value')['f1'].agg(['mean', 'std']).round(4).to_string())

## Phase 7 — Generate Figures (5, 6, 7)

In [ ]:
os.makedirs(os.path.join(REPO, 'figures/output'), exist_ok=True)

for fig_script in ['figures/figure5.py', 'figures/figure6.py', 'figures/figure7.py']:
    print(f'Generating {fig_script} …')
    r = subprocess.run([sys.executable, fig_script], cwd=REPO,
                       capture_output=True, text=True)
    out = (r.stdout + r.stderr).strip()
    print(out if out else '  (no output)')

print('\n✅ Figures saved to figures/output/')

In [ ]:
# Display figures inline (converts PDF → PNG via poppler)
import glob as _g, matplotlib.pyplot as plt, matplotlib.image as mpimg

!apt-get install -qq poppler-utils

for pdf in sorted(_g.glob(os.path.join(REPO, 'figures/output/*.pdf'))):
    stem = pdf.replace('.pdf', '')
    png  = stem + '.png'
    os.system(f'pdftoppm -r 150 -png "{pdf}" "{stem}" && mv "{stem}-1.png" "{png}" 2>/dev/null || true')
    if os.path.exists(png):
        print(f'\n─── {os.path.basename(pdf)} ───')
        img = mpimg.imread(png)
        plt.figure(figsize=(14, 5))
        plt.imshow(img); plt.axis('off'); plt.tight_layout(); plt.show()

## Phase 8 — LaTeX-Ready Tables

In [ ]:
os.makedirs(os.path.join(REPO, 'results/aggregated'), exist_ok=True)
summary.to_csv(os.path.join(REPO, 'results/aggregated/baselines.csv'))
summary_abl.to_csv(os.path.join(REPO, 'results/aggregated/ablation.csv'))
df_sens.to_csv(os.path.join(REPO, 'results/aggregated/sensitivity_raw.csv'), index=False)

MODEL_LABELS = {
    'static_gcn': 'Static GCN',
    'cnn': 'CNN-1D',
    'isolation_forest': 'Isolation Forest',
    'tgat': 'TGAT',
    'tgn': 'TGN',
    'tgnn': 'T-GNN (Ours)',
}
print('=== Table 2: Baseline Comparison (LaTeX) ===')
for model, row in summary.iterrows():
    lbl = MODEL_LABELS.get(model, model)
    print(f"{lbl:<22} & {row.Precision_mean:.4f} & {row.Recall_mean:.4f} "
          f"& {row.F1_mean:.4f}$\\pm${row.F1_std:.4f} "
          f"& {row.AUC_mean:.4f}$\\pm${row.AUC_std:.4f} \\\\")

ABL_LABELS = {
    'A1_static_gcn': 'A1: Static GCN',
    'A2_gcn_gru': 'A2: GCN+GRU',
    'A3_gcn_attention': 'A3: GCN+Attention',
    'A4_temporal_attention_no_hetero': 'A4: Temporal+Attn (homog.)',
    'A5_full_tgnn': 'A5: Full T-GNN (Ours)',
}
print('\n=== Table 3: Ablation Study (LaTeX) ===')
for config, row in summary_abl.iterrows():
    lbl = ABL_LABELS.get(config, config)
    print(f"{lbl:<35} & {row.F1_mean:.4f}$\\pm${row.F1_std:.4f} "
          f"& {row.AUC_mean:.4f}$\\pm${row.AUC_std:.4f} \\\\")

## Phase 9 — Download All Results

In [ ]:
import zipfile
from google.colab import files

ZIP_OUT = '/content/tgnn_results.zip'
PACK    = ['results/raw', 'results/aggregated', 'figures/output']

with zipfile.ZipFile(ZIP_OUT, 'w', zipfile.ZIP_DEFLATED) as zf:
    for folder in PACK:
        full_folder = os.path.join(REPO, folder)
        if not os.path.exists(full_folder):
            continue
        for root, _, fnames in os.walk(full_folder):
            for fname in fnames:
                fpath = os.path.join(root, fname)
                arc   = os.path.relpath(fpath, REPO)
                zf.write(fpath, arc)

size_mb = os.path.getsize(ZIP_OUT) / 1e6
print(f'✅ Ready: {ZIP_OUT}  ({size_mb:.1f} MB)')
files.download(ZIP_OUT)

---
## 🚀 One-Click Full Pipeline

After completing **Phase 1** (install) and **Phase 2** (upload), run this single cell instead of Phases 3–7:

In [ ]:
!python experiments/run_all.py \
    --seeds 42 123 2024 3407 7777 \
    --config configs/default.yaml \
    --seed-for-data 42 \
    2>&1 | tee results/logs/full_run.log

print('\n✅ Full pipeline complete — run Phase 9 to download results.')